# 00 · ElasticNet LOO result files

This notebook prepares the two ElasticNet LOO files in `../data/` (= `02. elasticNet/data/`):

| File | Content | Used by |
|---|---|---|
| `enet_all_results.h5` | **One row per protein × LOO sample** (`protein`, `LOO sample`, `true`, `predicted`, `params_dict`, …) | `01. res_analysis_loo`, `02_enet_scT_comparison` (Fig. S10a, S4) |
| `enet_best_results.h5` | **One row per protein × normalization**, the best LOO split of each `*_best` file (`best_sample`, `best_abs_error`, `mse_all_others`, `alpha`, `l1_ratio`, coefficients) | Summary / reference only |

How `enet_all_results.h5` is obtained (Step 1 picks the first case that applies):
1. **Already in `../data/`** (e.g. your original file) → it is validated and kept as is.
2. **`<P>_LOO.*` files** (new `new_pipeline.py --full_output`) or **`batch_*` files with per-sample rows** (original pipeline with `--full_output`) in `../data/loo/results_<NORMALIZATION>/` → it is built from them.
3. **Only `*_best` files** → it cannot be built: they hold one split per protein, not per-sample predictions. Re-run the LOO with `FULL_OUTPUT=1`, or put the original file in `../data/`.

Step 2 always builds `enet_best_results.h5` from whatever `*_best` files exist in `../data/loo/results_*/`.

In [1]:
from pathlib import Path
import ast, json
import numpy as np
import pandas as pd

DATA_DIR = Path('../data')
LOO_ROOT = DATA_DIR / 'loo'                     # results_zscore/, results_none/, results_ranking/, results_binning/
NORMALIZATION = 'zscore'                        # LOO run used for enet_all_results.h5 (paper: zscore)
LOO_DIR = LOO_ROOT / f'results_{NORMALIZATION}'

ALL_H5 = DATA_DIR / 'enet_all_results.h5'       # per-sample LOO predictions
BEST_H5 = DATA_DIR / 'enet_best_results.h5'     # best split per protein x normalization
ROW_COLS = {'protein', 'LOO sample', 'true', 'predicted'}

DATA_DIR.mkdir(parents=True, exist_ok=True)
print('Data folder :', DATA_DIR.resolve())
print('LOO folders :', sorted(p.name for p in LOO_ROOT.glob('results_*') if p.is_dir()) if LOO_ROOT.is_dir() else 'none')

Data folder : C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\02. elasticNet\data
LOO folders : ['results_binning', 'results_none', 'results_ranking', 'results_zscore']


In [2]:
# ---------- helpers ----------
def read_any(path):
    return pd.read_parquet(path) if path.suffix == '.parquet' else pd.read_csv(path)

def files_for(folder, pattern):
    """<pattern>.csv / .parquet in folder; parquet wins when both exist for the same name."""
    found = {}
    for f in sorted(folder.glob(pattern + '.csv')) + sorted(folder.glob(pattern + '.parquet')):
        found[f.name.rsplit('.', 1)[0]] = f
    return list(found.values())

def columns_of(path):
    if path.suffix == '.parquet':
        import pyarrow.parquet as pq
        return set(pq.read_schema(path).names)
    return set(pd.read_csv(path, nrows=0).columns)

def parse_dict(v):
    if isinstance(v, dict):
        return v
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return {}
    for parser in (json.loads, ast.literal_eval):
        try:
            out = parser(str(v))
            return out if isinstance(out, dict) else {}
        except Exception:
            pass
    return {}

def to_text(v):
    """Object cells -> plain strings, so HDF5 stores them without pickling (no PerformanceWarning)."""
    if isinstance(v, str):
        return v
    if isinstance(v, np.ndarray):
        return json.dumps(v.tolist())
    if isinstance(v, (dict, list, tuple)):
        return json.dumps(v, default=str)
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ''
    return str(v)

def h5_ready(df):
    df = df.copy()
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].map(to_text)
    return df

def read_h5_all_keys(path):
    with pd.HDFStore(path, mode='r') as store:      # exactly how notebooks 01 / 02 read it
        keys = store.keys()
        return pd.concat([store[k] for k in keys], ignore_index=True), keys

## Step 1 – `enet_all_results.h5` (per-sample LOO predictions)

In [3]:
ALL_STATUS = None

if ALL_H5.exists():
    # Case 1: keep the existing file (e.g. the original one) after checking it
    existing, keys = read_h5_all_keys(ALL_H5)
    missing = ROW_COLS - set(existing.columns)
    if missing:
        raise ValueError(f'{ALL_H5.name} exists but lacks columns needed by notebooks 01/02: {missing}. '
                         f'Columns found: {list(existing.columns)}')
    ALL_STATUS = 'existing'
    print(f'Kept existing {ALL_H5.name}: keys {keys}, {len(existing):,} rows, '
          f"{existing['protein'].nunique():,} proteins, {existing['LOO sample'].nunique():,} samples")
else:
    # Case 2: build it from per-sample rows (<P>_LOO files, or batch_* files of the original pipeline)
    source = files_for(LOO_DIR, '*_LOO') if LOO_DIR.is_dir() else []
    if not source and LOO_DIR.is_dir():
        source = [f for f in files_for(LOO_DIR, 'batch_*') if ROW_COLS <= columns_of(f)]
    if source:
        loo = pd.concat([read_any(f) for f in source], ignore_index=True)
        loo = loo.dropna(subset=['LOO sample', 'protein'])
        loo['true'] = pd.to_numeric(loo['true'], errors='coerce')
        loo['predicted'] = pd.to_numeric(loo['predicted'], errors='coerce')
        loo['protein'] = loo['protein'].astype(str)
        loo['LOO sample'] = loo['LOO sample'].astype(str)
        n_before = len(loo)
        loo = loo.drop_duplicates(['protein', 'LOO sample'], keep='last')
        if len(loo) < n_before:
            print(f'Dropped {n_before - len(loo):,} duplicated protein x sample rows (kept the latest).')
        h5_ready(loo).to_hdf(ALL_H5, key='data', mode='w', format='fixed')
        ALL_STATUS = 'built'
        print(f'Built {ALL_H5.name} from {len(source):,} files: {len(loo):,} rows, '
              f"{loo['protein'].nunique():,} proteins, {loo['LOO sample'].nunique():,} samples")
    else:
        # Case 3: only *_best files -> cannot be rebuilt; continue with Step 2
        n_best = len(files_for(LOO_DIR, '*_best')) if LOO_DIR.is_dir() else 0
        ALL_STATUS = 'missing'
        print(f'{ALL_H5.name} NOT available. {LOO_DIR} has no per-sample rows ({n_best:,} *_best files only).\n'
              '  -> put your original enet_all_results.h5 in ../data/, or re-run the LOO with FULL_OUTPUT=1.\n'
              '  Notebooks 01 and 02 need this file; Step 2 below still runs.')

Kept existing enet_all_results.h5: keys ['/A2M', '/AAAS', '/AACS', '/AAGAB', '/AAK1', '/AAMDC', '/AAMP', '/AAR2', '/AARS', '/AARS2', '/AARSD1', '/AASDHPPT', '/AASS', '/AATF', '/ABCB10', '/ABCB6', '/ABCB7', '/ABCB8', '/ABCC1', '/ABCC4', '/ABCD1', '/ABCD3', '/ABCE1', '/ABCF1', '/ABCF2', '/ABCF3', '/ABHD10', '/ABHD11', '/ABHD12', '/ABHD14B', '/ABHD16A', '/ABI1', '/ABI2', '/ABLIM1', '/ABT1', '/ACAA1', '/ACAA2', '/ACACA', '/ACAD10', '/ACAD8', '/ACAD9', '/ACADM', '/ACADS', '/ACADSB', '/ACADVL', '/ACAP2', '/ACAT1', '/ACAT2', '/ACBD3', '/ACBD5', '/ACBD6', '/ACIN1', '/ACLY', '/ACO1', '/ACO2', '/ACOT13', '/ACOT2', '/ACOT7', '/ACOT8', '/ACOT9', '/ACOX1', '/ACP1', '/ACP2', '/ACP6', '/ACSF2', '/ACSF3', '/ACSL1', '/ACSL3', '/ACSL4', '/ACSL5', '/ACSS2', '/ACTA2', '/ACTB', '/ACTL6A', '/ACTL8', '/ACTN1', '/ACTN4', '/ACTR10', '/ACTR1A', '/ACTR2', '/ACTR3', '/ACTR5', '/ACY1', '/ACYP1', '/ACYP2', '/ADA', '/ADAM10', '/ADAM15', '/ADAM17', '/ADAM9', '/ADAR', '/ADCK3', '/ADCK4', '/ADD1', '/ADD3', '/ADH5', '/A

## Step 2 – `enet_best_results.h5` (best split per protein, all normalizations)

In [4]:
frames = []
for folder in sorted(p for p in LOO_ROOT.glob('results_*') if p.is_dir()) if LOO_ROOT.is_dir() else []:
    files = files_for(folder, '*_best')
    if not files:
        continue
    df = pd.concat([read_any(f) for f in files], ignore_index=True)
    df['normalization'] = folder.name.replace('results_', '')
    params = df['params_dict'].map(parse_dict) if 'params_dict' in df else pd.Series([{}] * len(df))
    df['alpha'] = pd.to_numeric(params.map(lambda d: d.get('alpha', d.get('model__alpha', np.nan))), errors='coerce')
    df['l1_ratio'] = pd.to_numeric(params.map(lambda d: d.get('l1_ratio', d.get('model__l1_ratio', np.nan))), errors='coerce')
    if 'feat_coeffs_dict' in df:
        df['n_nonzero_coefs'] = df['feat_coeffs_dict'].map(parse_dict).map(
            lambda d: int(sum(1 for x in d.values() if x not in (0, 0.0, None))))
    frames.append(df)

if frames:
    best = pd.concat(frames, ignore_index=True)
    best['protein'] = best['protein'].astype(str)
    best = best.drop_duplicates(['normalization', 'protein'], keep='last')
    h5_ready(best).to_hdf(BEST_H5, key='data', mode='w', format='fixed')
    print(f'Wrote {BEST_H5.name}: {len(best):,} rows')
    summary = (best.groupby('normalization')
                   .agg(proteins=('protein', 'nunique'), median_mse_all_others=('mse_all_others', 'median'),
                        median_best_abs_error=('best_abs_error', 'median'),
                        median_alpha=('alpha', 'median'), median_l1_ratio=('l1_ratio', 'median'))
                   .round(4))
    display(summary)
else:
    print(f'No *_best files under {LOO_ROOT.resolve()}/results_*/ – {BEST_H5.name} not written.')

Wrote enet_best_results.h5: 6,678 rows


,proteins,median_mse_all_others,median_best_abs_error,median_alpha,median_l1_ratio
normalization,,,,,
binning,2141,0.2680,0.0012,0.5,0.5
none,2128,0.1866,0.0011,0.1,0.5
ranking,600,0.5434,0.0016,1.0,0.9
zscore,1809,0.1808,0.0011,0.1,0.5


## Step 3 – check both files exactly as the analysis notebooks read them

In [5]:
for path in (ALL_H5, BEST_H5):
    if path.exists():
        df, keys = read_h5_all_keys(path)
        print(f'{path.name:<22} OK  keys={keys}  shape={df.shape}  proteins={df["protein"].nunique():,}')
    else:
        print(f'{path.name:<22} not present')

if ALL_STATUS == 'missing':
    print('\nNotebooks 01. res_analysis_loo and 02_enet_scT_comparison will not run until enet_all_results.h5 is in ../data/.')
else:
    print(f'\nenet_all_results.h5 status: {ALL_STATUS}. Notebooks 01 and 02 can run.')

enet_all_results.h5    OK  keys=['/A2M', '/AAAS', '/AACS', '/AAGAB', '/AAK1', '/AAMDC', '/AAMP', '/AAR2', '/AARS', '/AARS2', '/AARSD1', '/AASDHPPT', '/AASS', '/AATF', '/ABCB10', '/ABCB6', '/ABCB7', '/ABCB8', '/ABCC1', '/ABCC4', '/ABCD1', '/ABCD3', '/ABCE1', '/ABCF1', '/ABCF2', '/ABCF3', '/ABHD10', '/ABHD11', '/ABHD12', '/ABHD14B', '/ABHD16A', '/ABI1', '/ABI2', '/ABLIM1', '/ABT1', '/ACAA1', '/ACAA2', '/ACACA', '/ACAD10', '/ACAD8', '/ACAD9', '/ACADM', '/ACADS', '/ACADSB', '/ACADVL', '/ACAP2', '/ACAT1', '/ACAT2', '/ACBD3', '/ACBD5', '/ACBD6', '/ACIN1', '/ACLY', '/ACO1', '/ACO2', '/ACOT13', '/ACOT2', '/ACOT7', '/ACOT8', '/ACOT9', '/ACOX1', '/ACP1', '/ACP2', '/ACP6', '/ACSF2', '/ACSF3', '/ACSL1', '/ACSL3', '/ACSL4', '/ACSL5', '/ACSS2', '/ACTA2', '/ACTB', '/ACTL6A', '/ACTL8', '/ACTN1', '/ACTN4', '/ACTR10', '/ACTR1A', '/ACTR2', '/ACTR3', '/ACTR5', '/ACY1', '/ACYP1', '/ACYP2', '/ADA', '/ADAM10', '/ADAM15', '/ADAM17', '/ADAM9', '/ADAR', '/ADCK3', '/ADCK4', '/ADD1', '/ADD3', '/ADH5', '/ADI1', '/